# EMULATE MY SIMPLE NCL ENSO DIAGNOSTICS BASED ON NINO3.4 TIMESERIES

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import linregress



import importlib

import os
os.cpu_count()

import enso_utils as mypy

In [ ]:
from dask.distributed import Client
from ncar_jobqueue import NCARCluster

In [ ]:
cluster = NCARCluster(project='P93300642',interface='ext',walltime="12:00:00")
cluster
cluster.scale(jobs=8)
client = Client(cluster)
client

## Setup Run Information

In [ ]:
''' CASE SPECIFICATIONS '''

#enso_cases = ['OBS','CESM2','CESM1','CESM3-beta']
#enso_names = [['ERA5','ERA5'],'b.e21.BHISTcmip6.f09_g17.LE2-1001.001','b.e11.B1850C5CN.f09_g16.005','b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.156']

enso_cases = ['CESM2','CESM1']
enso_names = ['b.e21.BHISTcmip6.f09_g17.LE2-1001.001','b.e11.B1850C5CN.f09_g16.005']


#enso_cases = ['OBS','CESM2','CESM1','156','162','163','166','170','171']
#enso_names = [['ERA5','TROPFLUX'], 
#              'b.e21.BHISTcmip6.f09_g17.LE2-1001.001',
#              'b.e11.B1850C5CN.f09_g16.005',
#              'b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.156',
#              'b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.162',
#              'b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.163',
#              'b.e30_beta06.B1850C_LTso.ne30_t232_wgx3.166',
#              'b.e30_beta06.B1850C_LTso.ne30_t232_wgx3.170',
#              'b.e30_beta06.B1850C_LTso.ne30_t232_wgx3.171'
#             ]              

#enso_cases = ['ERA5','156']
#enso_names = ['ERA5',
#              'b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.156']

xnino_reg = 'nino3' ; ynino_reg = 'nino3' ; 

#enso_ystart = [1979,1860,430,31]
#enso_yend   = [2018,1899,470,70]

enso_ystart = [1860,430,31]
enso_yend   = [1899,470,70]

#enso_ystart = [1979,1860,430,30,30,30,30,30,30]
#enso_yend   = [2018,1899,470,70,70,70,70,70,70]

#season = 'DJF' ; get_months = [12,1,2]
#season = 'NDJFM' ; get_months = [11,12,1,2,3]
season = 'ANN' ; get_months = [1,2,3,4,5,6,7,8,9,10,11,12]

lread_in_all_hist = False   # Read in all the CESM/CAM history files in a directory (max out for 100 years)

lwrite_ts_file = True  # Write out a timeseries of 2D SST/VAR variables (if they don't exist)
lread_ts_file = True # Read in a timeseries of 2D SST/VAR variables (if they exist).


ncases = len(enso_cases)

loop_icases = ncases # Number of set cases to run.
is_in_situ = False  # For real, non-reanlysis observations.

## Grab data and averaged

In [ ]:
''' VARIABLE SPECIFICATIONS '''

#x-var

#var_x = 'TS'  ; vunits_x = 'K'
var_x = 'PRECT' ; vunits_x = 'mm/day' # mon means (m->mm = *1000.)  
#var_y = 'TAUX'  ; vunits_y = 'N/m^2'
#var_comp = 'taux' ; evar_comp = 'chnk' ; ovar_scale = -30.*1.e3 ; cvar_comp = 'TAUX' ; cvar_comp = 'TAUX'; cvar_scale = -1.*1.e3 ;  vunits = 'N/m^2'

# y-var
#var_y = 'OMEGA500'  ; vunits_y = 'mb/hr'
#var_y = 'PRECT'DTC ; vunits_y = 'mm/day' # mon means (m->mm = *1000.)  
#var_y = 'TAUX' ; vunits_y = 'N/m^2'
var_y = 'DTCOND300'  ; vunits_y = 'K/day'

In [ ]:
importlib.reload(mypy)

dir_fig = '/glade/u/home/rneale/python/python-figs/enso/'



### Settings

figp, axp = plt.subplots(
    nrows=1,
    ncols=2,
    constrained_layout=True,
    figsize=(22, 10),
    )

figs, axs = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(12, 12),
    )


plt.rcdefaults()
plt.rcParams.update({'font.size': 18})


# Plot ranges

xmin,xmax,x_avals = mypy.fig_domains(var_x)
ymin,ymax,y_avals = mypy.fig_domains(var_y)


#scat_cols = ['black','blue','red','purple','orange','green','cyan']

scat_cols = [
    "red",
    "royalblue",
    "darkorange",
    "forestgreen",
    "firebrick",
    "goldenrod",
    "mediumpurple",
    "deepskyblue",
    "crimson"
    
]


# Analysis summary

print('### Analyzing ',var_y,' nino anomaly dependence on ',var_x,' nino anomlaies') 

''''''
''' LOOP OVER CASES '''
''''''

for icase,case in enumerate(enso_cases[0:loop_icases]):


#    lstyle = '--' if icase==0  else '-'
    lstyle = '-.' if icase in [0,1]  else '-'
    
    lmark = '^' if icase==0 else ''
        
    
    cname = enso_names[icase]
    
    
    yr0 = enso_ystart[icase] 
    yr1 = enso_yend[icase] 



    
    print('')
    print([icase+1],' of ',[ncases],' +++ ' , enso_names[icase] , ' +++')
    print('-- Reading in data --')


   
    
    # Case 'type'

    ctype = 'OBS'
    if 'b.e3' in cname : ctype = 'cesm3'
    if 'b.e2' in cname : ctype = 'cesm2'
    if 'b.e1' in cname : ctype = 'cesm1'

    
    
    # 1. Grab data for x-axis and y-axis variable (obs. or model)

    print('')
    print('** 1. Grabbing Case Data')

    # Some manipulation becasue of obs. could be different types ('cases').
    cname_xy = cname
    if not isinstance(cname, list): cname_xy = [cname]
        

    da_x = mypy.get_dataset(cname_xy[0],ctype,var_x,yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)
    da_y = mypy.get_dataset(cname_xy[-1],ctype,var_y,yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)

  
    # 2. Time info check

    print('')
    print('** 2. Checking and Trimming Time Dimensions')
    da_x, da_y = mypy.dataset_ts_trim(da_x, da_y, yr0, yr1, var_x, var_y, get_months)
    
    # Calculate nino3.4 and VAR timeseries.
    print('')
    print('** 3. Calculating Nino Anomalies')

    var_x_1d, var_x_pdf = mypy.nino_anom_ts(da_x,xnino_reg,x_avals)
    var_y_1d, var_y_pdf = mypy.nino_anom_ts(da_y,ynino_reg,y_avals)
    

   


    
    ''' Scatter plot of the relationship between SST and VAR '''    
    
    # Linear regression

    print('')
    print('** 4. Plotting')


    
    axp[0].plot(x_avals, var_x_pdf, linewidth=5, color=scat_cols[icase],linestyle=lstyle, label=case)
    axp[1].plot(y_avals, var_y_pdf, linewidth=5, color=scat_cols[icase],linestyle=lstyle)
    
    
    slope_neg, intercept_neg, r_value_neg, p_value_neg, std_err_neg = linregress(var_x_1d[var_x_1d < 0], var_x_1d[var_x_1d < 0])
    slope_pos, intercept_pos, r_value_pos, p_value_pos, std_err_pos = linregress(var_x_1d[var_x_1d > 0], var_x_1d[var_x_1d > 0])
    slope, intercept, r_value, p_value, std_err = linregress(var_x_1d, var_y_1d)
    
    varx_even_neg = np.linspace(xmin, 0, 50) 
    line_neg = slope_neg * varx_even_neg + intercept_neg

    varx_even_pos = np.linspace(0, xmax, 50) 
    line_pos = slope_pos * varx_even_pos + intercept_pos

    varx_even = np.linspace(xmin, xmax, 100) 
    line = slope * varx_even + intercept
    
    # Scatter plot

    case_yrs = case+' ['+str(int(yr0))+'-'+str(int(yr1))+']- '+f"{slope:.3g}"
    axs.scatter(var_x_1d, var_y_1d, label=case_yrs, alpha=0.7, color=scat_cols[icase],s=15)

    axs.plot(varx_even, line, color=scat_cols[icase],linewidth=4,linestyle=lstyle)
#    axs.plot(sst_even_pos, line_pos, color=scat_cols[icase],linewidth=4,linestyle=lstyle)

    
    axs.set_xlabel(var_x+' ('+vunits_x+')') ; axs.set_xlim([xmin,xmax])
    axs.set_ylabel(var_y+' ('+vunits_y+')') ; axs.set_ylim([ymin,ymax])
    axs.set_title('Linear Regression '+var_x+' ('+xnino_reg+') with '+var_y+' ('+ynino_reg+') - '+season)

    if icase==ncases-1:
        legend = axs.legend(title='Case/Years/Slope')
        legend_labels = legend.get_texts()
        legend.get_title().set_fontweight('bold') 
        for ileg, leg_lab in enumerate(legend_labels): leg_lab.set_color(scat_cols[ileg])   # Dataset 1
        
    axs.grid(True)
    axs.axhline(0, color='gray', linestyle='--', linewidth=2.5)
    axs.axvline(0, color='gray', linestyle='--', linewidth=2.5)

         
    del(da_x)
    del(da_y)




    
    # Set the PDF plots
    
    axp[0].set_title(var_x+' PDF ('+xnino_reg+')') ; axp[1].set_title(var_y+' PDF ('+ynino_reg+')') 
    axp[0].set_xlabel(vunits_x) ; axp[1].set_xlabel(vunits_y) 
    axp[0].set_ylabel('Density') ; axp[1].set_ylabel('Density')
    axp[0].grid(True) ; axp[1].grid(True) 
    axp[0].axvline(x=0., color='gray',linestyle='--', linewidth=2.5) ; axp[1].axvline(x=0., color='gray',linestyle='--', linewidth=2.5)
    axp[0].legend() ; axp[0].legend() 




# OUTPUT

figs.show()
figs.savefig(dir_fig+var_x+'_'+var_y+'_'+season+'_'+xnino_reg+'_'+ynino_reg+'_scatter_cesm3_monthly.png')
figp.savefig(dir_fig+var_x+'_'+var_y+'_'+season+'_'+xnino_reg+'_'+ynino_reg+'_1D_PDFs_cesm3_monthly.png')